<a href="https://colab.research.google.com/github/mujaffarattarai/Capstone/blob/main/StreamLit_Dashboard_with_USecase_1%2C2%2C3%2C4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Compliance Intelligence Platform
## Enterprise Dashboard — Google Colab Launcher

---

### 📋 Scenarios

| # | Scenario | Question |
|---|----------|----------|
| **1** | Recurring Non-Compliance | Which accounts / markets are likely to have recurring non-compliance in **2026 Q2**? |
| **2** | Systemic Root Causes | Which root causes will become systemic across more markets in **2026 Q2**? |

---

### 🚀 Quick Start

| Step | Action | Cell |
|------|--------|------|
| 1 | Install dependencies | Cell 1 |
| 2 | Mount Google Drive | Cell 2 |
| 3 | Write dashboard app to disk | Cell 3 |
| 4 | Paste ngrok token & launch | Cell 4 |

> 🔑 **Get a FREE ngrok token:** [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)


---
## 📦 Cell 1 — Install Dependencies


In [3]:
# ─── Install Dependencies ──────────────────────────────────────
!pip install -q streamlit pyngrok plotly openpyxl xlsxwriter scikit-learn

print("✅ All packages installed successfully")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 111.7 MB/s eta 0:00:00
✅ All packages installed successfully


---
## 💾 Cell 2 — Mount Google Drive


In [ ]:
# ─── Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

print("✅ Google Drive mounted at /content/drive")


---
## 📝 Cell 3 — Write Dashboard App to Disk

This cell uses  to save the full Streamlit dashboard as a Python file.  
The app includes:
- **Data upload & processing** with root-cause classification
- **Feature engineering** (rolling-window, zero data-leakage)
- **5 ML classifiers** with cross-validation
- **Interactive predictions** for 2026 Q2
- **Export** to Excel


In [15]:
%%writefile /content/compliance_dashboard.py
"""
Compliance Intelligence Platform - Enterprise Dashboard
Scenario 1: Recurring Non-Compliance | Scenario 2: Systemic Risk

HOW TO RUN:
    pip install streamlit plotly pandas numpy scikit-learn openpyxl xlsxwriter
    streamlit run compliance_dashboard.py
"""

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import io
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model          import LogisticRegression
from sklearn.tree                   import DecisionTreeClassifier
from sklearn.ensemble               import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm                    import SVC
from sklearn.naive_bayes            import MultinomialNB
from sklearn.calibration            import CalibratedClassifierCV
from sklearn.preprocessing          import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection        import (train_test_split, StratifiedKFold,
                                            cross_validate, cross_val_predict, LeaveOneOut)
from sklearn.metrics                import (accuracy_score, precision_score, recall_score,
                                            f1_score, roc_auc_score, roc_curve,
                                            confusion_matrix, classification_report)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster                 import KMeans
from sklearn.inspection              import permutation_importance
from sklearn.base                    import clone
from sklearn.svm                     import LinearSVC

# ═══════════════════════════════════════════════════════════════
#  PAGE CONFIG
# ═══════════════════════════════════════════════════════════════
st.set_page_config(
    page_title="Compliance Intelligence Platform",
    page_icon="\U0001f6e1\ufe0f",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ═══════════════════════════════════════════════════════════════
#  THEME - Clean light palette
# ═══════════════════════════════════════════════════════════════
BG_MAIN   = "#f8f9fc"
BG_CARD   = "#ffffff"
BORDER    = "#e2e5ed"
ACCENT    = "#4361ee"
GREEN     = "#06d6a0"
ORANGE    = "#fb8500"
RED       = "#ef476f"
PURPLE    = "#7b2cbf"
TEAL      = "#118ab2"
TEXT_PRI  = "#1a1a2e"
TEXT_SEC  = "#6b7280"

PALETTE = [ACCENT, GREEN, ORANGE, RED, PURPLE, TEAL, "#e63946", "#f4a261", "#457b9d"]

CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

html, body, [data-testid="stAppViewContainer"] {{
    background: {BG_MAIN};
    color: {TEXT_PRI};
    font-family: 'DM Sans', -apple-system, BlinkMacSystemFont, sans-serif;
}}
[data-testid="stSidebar"] {{
    background: {BG_CARD} !important;
    border-right: 1px solid {BORDER};
}}
[data-testid="stHeader"] {{ background: transparent; }}

[data-testid="stMetric"] {{
    background: {BG_CARD};
    border: 1px solid {BORDER};
    border-radius: 12px;
    padding: 16px 20px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04);
}}
[data-testid="stMetricLabel"] {{
    color: {TEXT_SEC} !important;
    font-size: 11px;
    letter-spacing: 0.5px;
    text-transform: uppercase;
    font-weight: 600;
}}
[data-testid="stMetricValue"] {{
    color: {TEXT_PRI} !important;
    font-size: 28px;
    font-weight: 700;
}}

[data-testid="stTabs"] [data-baseweb="tab-list"] {{
    background: {BG_CARD};
    border-radius: 12px 12px 0 0;
    border-bottom: 1px solid {BORDER};
    gap: 2px;
    padding: 6px;
}}
[data-baseweb="tab"] {{
    color: {TEXT_SEC};
    font-size: 13px;
    font-weight: 500;
    padding: 10px 20px;
    border-radius: 8px;
}}
[aria-selected="true"][data-baseweb="tab"] {{
    color: {ACCENT} !important;
    background: {BG_MAIN} !important;
    font-weight: 600;
    border-bottom: 2px solid {ACCENT} !important;
}}

[data-testid="baseButton-primary"] {{
    background: {ACCENT} !important;
    border: none;
    color: #fff !important;
    font-weight: 600;
    border-radius: 8px;
    padding: 8px 24px;
    box-shadow: 0 2px 8px rgba(67, 97, 238, 0.3);
}}
[data-testid="baseButton-secondary"] {{
    background: transparent !important;
    border: 1px solid {BORDER} !important;
    color: {TEXT_PRI} !important;
    border-radius: 8px;
}}

[data-testid="stDataFrame"] {{
    border: 1px solid {BORDER};
    border-radius: 12px;
    overflow: hidden;
}}
[data-testid="stExpander"] {{
    border: 1px solid {BORDER} !important;
    border-radius: 12px !important;
    background: {BG_CARD} !important;
}}
[data-baseweb="select"] {{ background: {BG_CARD}; border-color: {BORDER}; border-radius: 8px; }}

::-webkit-scrollbar {{ width: 6px; height: 6px; }}
::-webkit-scrollbar-track {{ background: {BG_MAIN}; }}
::-webkit-scrollbar-thumb {{ background: {BORDER}; border-radius: 3px; }}
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════════
#  CONSTANTS
# ═══════════════════════════════════════════════════════════════
VALID_QS  = ['2024 Q1','2024 Q2','2024 Q3','2024 Q4',
             '2025 Q1','2025 Q2','2025 Q3','2025 Q4','2026 Q1']
FUTURE_QS = ['2026 Q2','2026 Q3','2026 Q4']

S1_FEATURES = ['total_obs','quarters_active','unique_domains',
               'unique_processes','recent_obs','streak','market_enc','category_enc']
S2_FEATURES = ['curr_markets','total_markets_hist','total_obs_hist',
               'quarters_active','unique_processes','rolling_mean_mkts',
               'rolling_std_mkts','lag1_mkts','lag2_mkts','trend_mkts',
               'obs_curr_q','streak']

MODELS_DEF = {
    'Logistic Regression' : lambda: LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree'       : lambda: DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
    'Random Forest'       : lambda: RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'Gradient Boosting'   : lambda: GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM'                 : lambda: SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
}
SCALED_MODELS = {'Logistic Regression', 'SVM'}

# ═══════════════════════════════════════════════════════════════
#  ROOT CAUSE CLASSIFIER
# ═══════════════════════════════════════════════════════════════
def classify_root_cause(text):
    text = str(text).lower()
    if any(w in text for w in ["not aware","lack of awareness","training not provided",
                                "training required","user unaware","misunderstood","incorrectly followed"]):
        return "Education Gap"
    if "policy" in text and any(w in text for w in ["not defined","missing","outdated"]):
        return "Policy Gap"
    if any(w in text for w in ["not followed","not performed","not documented","no evidence","not completed"]):
        return "Process Gap"
    if "access" in text and any(w in text for w in ["not reviewed","not removed","not revoked","excessive","unauthorized"]):
        return "Access Control Gap"
    if any(w in text for w in ["security","firewall","encryption","malware","vulnerability","patch","antivirus"]):
        return "Security Management"
    if any(w in text for w in ["not reported","not escalated","no notification","no alert"]):
        return "Reporting Gap"
    return "Execution Gap"

def apply_kmeans_subclusters(df):
    df = df.copy()
    df['Root_Cause_Type'] = df['Observation'].apply(classify_root_cause)
    sec_mask = df['Root_Cause_Type'] == 'Security Management'
    sec_df = df[sec_mask]
    if len(sec_df) >= 5:
        try:
            vec = TfidfVectorizer(max_features=100, stop_words='english')
            tfidf = vec.fit_transform(sec_df['Observation'].fillna(''))
            km = KMeans(n_clusters=5, random_state=42, n_init=10)
            km.fit(tfidf)
            labels_map = {
                0: "Logging & Retention Gap",
                1: "Firewall Rule Governance Gap",
                2: "Excessive Privilege Enforcement Gap",
                3: "Malware Defense Control Gap",
                4: "Security Review Not Performed"
            }
            df.loc[sec_mask, 'Root_Cause_Type'] = [labels_map[l] for l in km.labels_]
        except Exception:
            pass
    df['Root_Cause_Predicted'] = df['Root_Cause_Type']
    return df

# ═══════════════════════════════════════════════════════════════
#  DATA LOADING
# ═══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def load_and_clean(file_bytes):
    df_raw = pd.read_excel(io.BytesIO(file_bytes))
    df = df_raw[df_raw['FQ'].isin(VALID_QS)].copy()
    df = apply_kmeans_subclusters(df)
    return df

# ═══════════════════════════════════════════════════════════════
#  SCENARIO 1 - Feature Engineering
# ═══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def build_s1_samples(_df):
    le_mkt = LabelEncoder()
    le_cat = LabelEncoder()
    _df = _df.copy()
    _df['market_enc'] = le_mkt.fit_transform(_df['Market'].astype(str))
    _df['category_enc'] = le_cat.fit_transform(_df['Category'].astype(str).fillna('Unknown'))

    samples = []
    for acc in _df['Account'].unique():
        acc_df = _df[_df['Account'] == acc]
        acc_qs = set(acc_df['FQ'].unique())
        market = acc_df['Market'].mode()[0]
        cat = acc_df['Category'].mode()[0]
        m_enc = acc_df['market_enc'].mode()[0]
        c_enc = acc_df['category_enc'].mode()[0]

        for t_idx in range(2, len(VALID_QS)):
            target_q = VALID_QS[t_idx]
            history_q = VALID_QS[:t_idx]
            hist_df = acc_df[acc_df['FQ'].isin(history_q)]
            if len(hist_df) == 0:
                continue

            streak = 0
            for q in reversed(history_q):
                if q in acc_qs: streak += 1
                else: break

            samples.append({
                'Account': acc, 'market': market, 'category': cat,
                'total_obs': len(hist_df),
                'quarters_active': hist_df['FQ'].nunique(),
                'unique_domains': hist_df['Domain Area'].nunique() if 'Domain Area' in hist_df else 1,
                'unique_processes': hist_df['Process'].nunique() if 'Process' in hist_df else 1,
                'recent_obs': len(acc_df[acc_df['FQ'] == history_q[-1]]),
                'streak': streak,
                'market_enc': m_enc, 'category_enc': c_enc,
                'target_q': target_q,
                'recurring': int(target_q in acc_qs),
            })
    return pd.DataFrame(samples)

# ═══════════════════════════════════════════════════════════════
#  SCENARIO 2 - Feature Engineering
# ═══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def build_s2_samples(_df, threshold=3):
    samples = []
    root_causes = _df['Root_Cause_Predicted'].unique()

    for rc in root_causes:
        rc_df = _df[_df['Root_Cause_Predicted'] == rc]
        for t_idx in range(2, len(VALID_QS)):
            target_q = VALID_QS[t_idx]
            history_q = VALID_QS[:t_idx]
            hist_df = rc_df[rc_df['FQ'].isin(history_q)]
            if len(hist_df) == 0:
                continue

            curr_q_df = rc_df[rc_df['FQ'] == history_q[-1]]
            curr_mkts = curr_q_df['Market'].nunique()
            hist_mkts_prev = [rc_df[rc_df['FQ'] == q]['Market'].nunique() for q in history_q]

            active = [x for x in hist_mkts_prev if x > 0]
            lag1 = active[-1] if len(active) >= 1 else 0
            lag2 = active[-2] if len(active) >= 2 else 0

            streak = 0
            for q in reversed(history_q):
                if rc_df[rc_df['FQ'] == q]['Market'].nunique() > 0: streak += 1
                else: break

            tgt_df = rc_df[rc_df['FQ'] == target_q]
            tgt_new_mkts = len(set(tgt_df['Market'].unique()) - set(hist_df['Market'].unique()))
            systemic = int(tgt_new_mkts >= threshold)

            samples.append({
                'Root_Cause': rc,
                'curr_markets': curr_mkts,
                'total_markets_hist': hist_df['Market'].nunique(),
                'total_obs_hist': len(hist_df),
                'quarters_active': hist_df['FQ'].nunique(),
                'unique_processes': hist_df['Process'].nunique() if 'Process' in hist_df else 1,
                'rolling_mean_mkts': np.mean(active) if active else 0,
                'rolling_std_mkts': np.std(active) if len(active) > 1 else 0,
                'lag1_mkts': lag1, 'lag2_mkts': lag2,
                'trend_mkts': lag1 - lag2,
                'obs_curr_q': len(curr_q_df),
                'streak': streak,
                'target_q': target_q,
                'systemic': systemic,
            })
    return pd.DataFrame(samples)

# ═══════════════════════════════════════════════════════════════
#  MODEL TRAINING - Scenario 1
# ═══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def train_s1_models(_samples_df, selected_models):
    df = _samples_df.dropna(subset=S1_FEATURES + ['recurring'])
    X = df[S1_FEATURES].values
    y = df['recurring'].values

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    sc = StandardScaler().fit(X_tr)
    X_tr_sc = sc.transform(X_tr)
    X_te_sc = sc.transform(X_te)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = {}

    for name in selected_models:
        mdl = MODELS_DEF[name]()
        Xtr_use = X_tr_sc if name in SCALED_MODELS else X_tr
        Xte_use = X_te_sc if name in SCALED_MODELS else X_te

        cv = cross_validate(mdl, Xtr_use, y_tr, cv=skf, scoring=['f1','roc_auc'], return_train_score=False)
        mdl.fit(Xtr_use, y_tr)

        y_pred = mdl.predict(Xte_use)
        y_proba = mdl.predict_proba(Xte_use)[:, 1]
        fpr, tpr, _ = roc_curve(y_te, y_proba)

        results[name] = {
            'cv_f1': cv['test_f1'].mean(), 'cv_f1_std': cv['test_f1'].std(),
            'cv_auc': cv['test_roc_auc'].mean(),
            'accuracy': accuracy_score(y_te, y_pred),
            'precision': precision_score(y_te, y_pred, zero_division=0),
            'recall': recall_score(y_te, y_pred, zero_division=0),
            'f1': f1_score(y_te, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_te, y_proba),
            'cm': confusion_matrix(y_te, y_pred).tolist(),
            'fpr': fpr.tolist(), 'tpr': tpr.tolist(),
            'model': mdl,
            'composite': roc_auc_score(y_te, y_proba)*0.5 + recall_score(y_te, y_pred, zero_division=0)*0.3 + cv['test_f1'].mean()*0.2,
        }

    best_name = max(results, key=lambda k: results[k]['composite'])
    best_mdl = results[best_name]['model']
    Xte_best = X_te_sc if best_name in SCALED_MODELS else X_te

    if hasattr(best_mdl, 'feature_importances_'):
        fi = best_mdl.feature_importances_
    else:
        pi = permutation_importance(best_mdl, Xte_best, y_te, n_repeats=10, random_state=42)
        fi = pi.importances_mean

    feat_imp = dict(zip(S1_FEATURES, fi))
    return results, feat_imp, df, sc, best_name, X_tr, X_tr_sc, X_te, X_te_sc, y_tr, y_te

# ═══════════════════════════════════════════════════════════════
#  MODEL TRAINING - Scenario 2
# ═══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def train_s2_models(_samples_df, selected_models):
    df = _samples_df.copy()
    rc_counts = df.groupby('Root_Cause').size()
    valid_rcs = rc_counts[rc_counts >= 2].index
    df = df[df['Root_Cause'].isin(valid_rcs)]

    if len(df) < 2:
        st.warning("Not enough samples for Scenario 2 training. Try lowering the systemic threshold.")
        return {}, {}, df, StandardScaler(), ""

    X = df[S2_FEATURES].fillna(0).values
    y = df['systemic'].values

    sc = StandardScaler().fit(X)
    X_sc = sc.transform(X)
    loo = LeaveOneOut()

    results = {}
    for name in selected_models:
        preds, probas = [], []
        for tr_idx, te_idx in loo.split(X):
            mdl = MODELS_DEF[name]()
            Xu = X_sc if name in SCALED_MODELS else X
            mdl.fit(Xu[tr_idx], y[tr_idx])
            preds.append(mdl.predict(Xu[te_idx])[0])
            probas.append(mdl.predict_proba(Xu[te_idx])[0, 1])

        preds = np.array(preds)
        probas = np.array(probas)
        fpr, tpr, _ = roc_curve(y, probas)
        cm = confusion_matrix(y, preds)

        results[name] = {
            'loo_f1': f1_score(y, preds, zero_division=0),
            'loo_recall': recall_score(y, preds, zero_division=0),
            'loo_auc': roc_auc_score(y, probas),
            'cm': cm.tolist(), 'fpr': fpr.tolist(), 'tpr': tpr.tolist(),
            'composite': roc_auc_score(y, probas)*0.5 + recall_score(y, preds, zero_division=0)*0.3 + f1_score(y, preds, zero_division=0)*0.2,
            'y_all': y, 'y_pred': preds, 'y_proba': probas,
        }

    best_name = max(results, key=lambda k: results[k]['composite'])
    best_mdl = MODELS_DEF[best_name]()
    Xu = X_sc if best_name in SCALED_MODELS else X
    best_mdl.fit(Xu, y)
    results[best_name]['final_model'] = best_mdl

    if hasattr(best_mdl, 'feature_importances_'):
        fi = best_mdl.feature_importances_
    else:
        pi = permutation_importance(best_mdl, Xu, y, n_repeats=10, random_state=42)
        fi = pi.importances_mean

    feat_imp = dict(zip(S2_FEATURES, fi))
    return results, feat_imp, df, sc, best_name

# ═══════════════════════════════════════════════════════════════
#  PREDICTIONS - Scenario 1
# ═══════════════════════════════════════════════════════════════
def predict_s1(_df, model_name, trained_results, sc_obj, samples_df):
    mdl = trained_results[model_name]['model']
    le_mkt = LabelEncoder().fit(_df['Market'].astype(str))
    le_cat = LabelEncoder().fit(_df['Category'].astype(str).fillna('Unknown'))

    preds = []
    for acc in _df['Account'].unique():
        acc_df = _df[_df['Account'] == acc]
        acc_qs = set(acc_df['FQ'].unique())
        streak = 0
        for q in reversed(VALID_QS):
            if q in acc_qs: streak += 1
            else: break

        row = {
            'Account': acc,
            'Market': acc_df['Market'].mode()[0],
            'total_obs': len(acc_df),
            'quarters_active': acc_df['FQ'].nunique(),
            'unique_domains': acc_df['Domain Area'].nunique() if 'Domain Area' in acc_df else 1,
            'unique_processes': acc_df['Process'].nunique() if 'Process' in acc_df else 1,
            'recent_obs': len(acc_df[acc_df['FQ'] == VALID_QS[-1]]),
            'streak': streak,
            'market_enc': int(le_mkt.transform([acc_df['Market'].mode()[0]])[0]) if acc_df['Market'].mode()[0] in le_mkt.classes_ else 0,
            'category_enc': int(le_cat.transform([str(acc_df['Category'].mode()[0])])[0]) if str(acc_df['Category'].mode()[0]) in le_cat.classes_ else 0,
        }
        preds.append(row)

    pred_df = pd.DataFrame(preds)
    X_pred = pred_df[S1_FEATURES].values
    if model_name in SCALED_MODELS:
        X_pred = sc_obj.transform(X_pred)

    pred_df['Probability'] = mdl.predict_proba(X_pred)[:, 1]
    pred_df['Prediction'] = (pred_df['Probability'] >= 0.5).astype(int)
    pred_df['Risk Level'] = pd.cut(pred_df['Probability'], bins=[0, 0.3, 0.6, 1.0], labels=['Low', 'Medium', 'High'])
    return pred_df.sort_values('Probability', ascending=False)

# ═══════════════════════════════════════════════════════════════
#  PREDICTIONS - Scenario 2
# ═══════════════════════════════════════════════════════════════
def predict_s2(_df, model_name, trained_results, sc_obj):
    if 'final_model' not in trained_results[model_name]:
        return pd.DataFrame()

    mdl = trained_results[model_name]['final_model']
    preds = []

    for rc in _df['Root_Cause_Predicted'].unique():
        rc_df = _df[_df['Root_Cause_Predicted'] == rc]
        curr_df = rc_df[rc_df['FQ'] == VALID_QS[-1]]
        if curr_df.empty:
            continue

        hist_mkts = [rc_df[rc_df['FQ'] == q]['Market'].nunique() for q in VALID_QS]
        active = [x for x in hist_mkts if x > 0]
        lag1 = active[-1] if len(active) >= 1 else 0
        lag2 = active[-2] if len(active) >= 2 else 0

        streak = 0
        for q in reversed(VALID_QS):
            if rc_df[rc_df['FQ'] == q]['Market'].nunique() > 0: streak += 1
            else: break

        preds.append({
            'Root_Cause': rc,
            'curr_markets': curr_df['Market'].nunique(),
            'total_markets_hist': rc_df['Market'].nunique(),
            'total_obs_hist': len(rc_df),
            'quarters_active': rc_df['FQ'].nunique(),
            'unique_processes': rc_df['Process'].nunique() if 'Process' in rc_df else 1,
            'rolling_mean_mkts': np.mean(active) if active else 0,
            'rolling_std_mkts': np.std(active) if len(active) > 1 else 0,
            'lag1_mkts': lag1, 'lag2_mkts': lag2,
            'trend_mkts': lag1 - lag2,
            'obs_curr_q': len(curr_df),
            'streak': streak,
        })

    pred_df = pd.DataFrame(preds)
    if pred_df.empty:
        return pred_df

    X_pred = pred_df[S2_FEATURES].fillna(0).values
    if model_name in SCALED_MODELS:
        X_pred = sc_obj.transform(X_pred)

    pred_df['Probability'] = mdl.predict_proba(X_pred)[:, 1]
    pred_df['Systemic'] = (pred_df['Probability'] >= 0.5).astype(int)
    pred_df['Risk Level'] = pd.cut(pred_df['Probability'], bins=[0, 0.3, 0.6, 1.0], labels=['Low', 'Medium', 'High'])
    return pred_df.sort_values('Probability', ascending=False)


# ═══════════════════════════════════════════════════════════════
#  SCENARIO 3 — Risk Escalation (Market-level)
# ═══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def build_s3_market_q(_df):
    """Build market × quarter feature matrix with Enforcement Risk Score."""
    _df = _df.copy()
    _df['Tool_Gap_Flag'] = _df['Observation'].str.contains(
        'manual|spreadsheet|no tool|no automation', case=False, na=False).astype(int)
    _df['EOL_Flag'] = _df['Observation'].str.contains(
        'eol|end of life', case=False, na=False).astype(int)

    mq = _df.groupby(['Market','FQ']).agg(
        Execution_Count=('Root_Cause_Predicted', lambda x: x.str.contains('Gap', na=False).sum()),
        Tool_Gap_Count=('Tool_Gap_Flag', 'sum'),
        EOL_Count=('EOL_Flag', 'sum'),
        Process_Spread=('Process', 'nunique'),
        Account_Count=('Account', 'nunique'),
        Obs_Count=('Account', 'count'),
    ).reset_index()

    sc = MinMaxScaler()
    feat_cols = ['Execution_Count','Tool_Gap_Count','EOL_Count','Process_Spread']
    mq[feat_cols] = sc.fit_transform(mq[feat_cols])

    mq['Enforcement_Risk_Score'] = (
        0.40*mq['Execution_Count'] + 0.25*mq['Tool_Gap_Count'] +
        0.20*mq['EOL_Count'] + 0.15*mq['Process_Spread'])

    # Percentile-based thresholds
    low_t = mq['Enforcement_Risk_Score'].quantile(0.50)
    high_t = mq['Enforcement_Risk_Score'].quantile(0.80)
    mq['Risk_Level'] = mq['Enforcement_Risk_Score'].apply(
        lambda s: 'Low' if s < low_t else ('Moderate' if s < high_t else 'High'))

    # Lag features
    mq = mq.sort_values(['Market','FQ']).reset_index(drop=True)
    mq['lag1_score'] = mq.groupby('Market')['Enforcement_Risk_Score'].shift(1)
    mq['trend_score'] = mq['Enforcement_Risk_Score'] - mq['lag1_score']

    # Escalation label
    mq['Next_Risk_Level'] = mq.groupby('Market')['Risk_Level'].shift(-1)
    order = {'Low': 0, 'Moderate': 1, 'High': 2}
    mq['Escalation_Flag'] = mq.apply(
        lambda r: 1 if (not pd.isna(r['Next_Risk_Level']) and
                        order.get(r['Next_Risk_Level'], 0) > order.get(r['Risk_Level'], 0))
                    else 0, axis=1)

    return mq, sc, low_t, high_t

S3_FEATURES = ['trend_score', 'Enforcement_Risk_Score', 'Execution_Count',
               'Tool_Gap_Count', 'EOL_Count', 'Process_Spread']

@st.cache_data(show_spinner=False)
def train_s3_models(_market_q):
    """Train Scenario 3: LOO CV with multiple models."""
    train_df = _market_q.dropna(subset=['Next_Risk_Level', 'lag1_score']).copy()
    if len(train_df) < 5:
        return {}, {}, train_df, StandardScaler(), ""

    X = train_df[S3_FEATURES].fillna(0).values
    y = train_df['Escalation_Flag'].values

    if y.sum() < 2:
        return {}, {}, train_df, StandardScaler(), ""

    sc = StandardScaler().fit(X)
    X_sc = sc.transform(X)
    loo = LeaveOneOut()

    s3_models = {
        'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=4, min_samples_leaf=2,
                                                 class_weight='balanced', random_state=42),
        'SVM': SVC(class_weight='balanced', probability=True, kernel='rbf', random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42),
    }
    THRESHOLD_S3 = 0.40

    results = {}
    for name, model in s3_models.items():
        loo_probs, loo_true = [], []
        for tr_idx, val_idx in loo.split(X_sc):
            m = clone(model)
            m.fit(X_sc[tr_idx], y[tr_idx])
            loo_probs.append(m.predict_proba(X_sc[val_idx])[0][1])
            loo_true.append(y[val_idx][0])

        probs_arr = np.array(loo_probs)
        true_arr = np.array(loo_true)
        preds_arr = (probs_arr >= THRESHOLD_S3).astype(int)
        fpr, tpr, _ = roc_curve(true_arr, probs_arr)

        results[name] = {
            'loo_f1': f1_score(true_arr, preds_arr, zero_division=0),
            'loo_recall': recall_score(true_arr, preds_arr, zero_division=0),
            'loo_auc': roc_auc_score(true_arr, probs_arr) if len(set(true_arr)) > 1 else 0.5,
            'cm': confusion_matrix(true_arr, preds_arr).tolist(),
            'fpr': fpr.tolist(), 'tpr': tpr.tolist(),
            'composite': (roc_auc_score(true_arr, probs_arr) if len(set(true_arr)) > 1 else 0.5)*0.5
                         + recall_score(true_arr, preds_arr, zero_division=0)*0.3
                         + f1_score(true_arr, preds_arr, zero_division=0)*0.2,
            'y_all': true_arr, 'y_pred': preds_arr, 'y_proba': probs_arr,
        }

    best_name = max(results, key=lambda k: results[k]['composite'])
    # Retrain best on full data
    best_mdl = clone(s3_models[best_name])
    best_mdl.fit(X_sc, y)
    results[best_name]['final_model'] = best_mdl

    fi = permutation_importance(best_mdl, X_sc, y, n_repeats=10, random_state=42, scoring='f1')
    feat_imp = dict(zip(S3_FEATURES, fi.importances_mean))

    return results, feat_imp, train_df, sc, best_name


def predict_s3(_market_q, model_name, trained_results, sc_obj, valid_qs):
    """Predict 2026 Q2 risk escalations."""
    if 'final_model' not in trained_results.get(model_name, {}):
        return pd.DataFrame()
    mdl = trained_results[model_name]['final_model']
    curr_q = valid_qs[-1]
    curr = _market_q[_market_q['FQ'] == curr_q].copy()
    if curr.empty:
        return pd.DataFrame()
    curr['lag1_score'] = curr['lag1_score'].fillna(curr['Enforcement_Risk_Score'])
    curr['trend_score'] = curr['trend_score'].fillna(0)

    X_pred = sc_obj.transform(curr[S3_FEATURES].fillna(0).values)
    curr['Probability'] = mdl.predict_proba(X_pred)[:, 1]
    curr['Predicted_Escalation'] = (curr['Probability'] >= 0.40).astype(int)
    return curr.sort_values('Probability', ascending=False)


# ═══════════════════════════════════════════════════════════════
#  SCENARIO 4 — Root Cause Prediction (Text Classification)
# ═══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def train_s4_models(_df):
    """Train multi-class text classifier for root cause prediction."""
    _df = _df.copy()
    texts = _df['Observation'].fillna('').values
    labels = _df['Root_Cause_Predicted'].values

    # Remove rare classes
    class_counts = pd.Series(labels).value_counts()
    valid_classes = class_counts[class_counts >= 10].index
    mask = pd.Series(labels).isin(valid_classes)
    texts = texts[mask]
    labels = labels[mask]

    if len(texts) < 20:
        return {}, None, None, None, ""

    le = LabelEncoder()
    y = le.fit_transform(labels)

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english', min_df=2)
    X = tfidf.fit_transform(texts)

    skf = StratifiedKFold(n_splits=min(10, min(pd.Series(y).value_counts())), shuffle=True, random_state=42)

    s4_models = {
        'Logistic Regression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42),
        'LinearSVC': CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=3000, random_state=42)),
        'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
        'Naive Bayes': MultinomialNB(alpha=1.0),
    }

    results = {}
    for name, model in s4_models.items():
        preds = cross_val_predict(model, X, y, cv=skf)
        probas = cross_val_predict(model, X, y, cv=skf, method='predict_proba')

        macro_f1 = f1_score(y, preds, average='macro', zero_division=0)
        acc = accuracy_score(y, preds)
        cm = confusion_matrix(y, preds)

        results[name] = {
            'macro_f1': macro_f1,
            'accuracy': acc,
            'cm': cm.tolist(),
            'composite': macro_f1 * 0.6 + acc * 0.4,
            'y_all': y, 'y_pred': preds,
        }

    best_name = max(results, key=lambda k: results[k]['composite'])
    # Retrain best on full data
    best_mdl = clone(s4_models[best_name])
    best_mdl.fit(X, y)

    return results, best_mdl, tfidf, le, best_name

# ═══════════════════════════════════════════════════════════════
#  CHART HELPERS
# ═══════════════════════════════════════════════════════════════
CHART_LAYOUT = dict(
    paper_bgcolor=BG_CARD, plot_bgcolor=BG_MAIN,
    font=dict(color=TEXT_PRI, family="DM Sans, sans-serif", size=12),
    margin=dict(l=20, r=20, t=48, b=20),
    legend=dict(bgcolor=BG_CARD, bordercolor=BORDER, borderwidth=1),
    xaxis=dict(gridcolor=BORDER, zerolinecolor=BORDER),
    yaxis=dict(gridcolor=BORDER, zerolinecolor=BORDER),
)

def chart_model_comparison(results, metric_cols, scenario):
    models = list(results.keys())
    fig = make_subplots(rows=1, cols=len(metric_cols), subplot_titles=metric_cols)
    for col_i, metric in enumerate(metric_cols, 1):
        vals = [results[m].get(metric, 0) for m in models]
        colors = [GREEN if v == max(vals) else ACCENT for v in vals]
        fig.add_trace(go.Bar(x=models, y=vals, marker_color=colors,
            text=[f"{v:.3f}" for v in vals], textposition='outside',
            textfont=dict(size=10, color=TEXT_SEC), name=metric, showlegend=False
        ), row=1, col=col_i)
        fig.update_yaxes(range=[0, 1.1], row=1, col=col_i)
    fig.update_layout(**CHART_LAYOUT, title=f"Model Comparison \u2014 {scenario}", height=350)
    return fig

def chart_roc_curves(results, title):
    fig = go.Figure()
    fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1, line=dict(color=TEXT_SEC, dash='dash', width=1))
    for i, (name, r) in enumerate(results.items()):
        fig.add_trace(go.Scatter(x=r['fpr'], y=r['tpr'], mode='lines',
            name=f"{name} (AUC={r.get('roc_auc', r.get('loo_auc', 0)):.3f})",
            line=dict(color=PALETTE[i % len(PALETTE)], width=2)))
    fig.update_layout(**CHART_LAYOUT, title=title, xaxis_title='FPR', yaxis_title='TPR', height=380)
    return fig

def chart_confusion_matrix(cm_list, model_name):
    cm = np.array(cm_list)
    fig = go.Figure(go.Heatmap(z=cm, x=['Pred 0','Pred 1'], y=['True 0','True 1'],
        colorscale=[[0, BG_MAIN],[1, ACCENT]], text=cm, texttemplate='%{text}',
        textfont=dict(size=20, color=TEXT_PRI), showscale=False))
    fig.update_layout(**CHART_LAYOUT, title=f"CM \u2014 {model_name}", height=280)
    fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
    return fig

def chart_feature_importance(feat_imp, title):
    sorted_fi = dict(sorted(feat_imp.items(), key=lambda x: x[1]))
    colors = [GREEN if v == max(feat_imp.values()) else ACCENT for v in sorted_fi.values()]
    fig = go.Figure(go.Bar(x=list(sorted_fi.values()), y=list(sorted_fi.keys()), orientation='h',
        marker_color=colors, text=[f"{v:.4f}" for v in sorted_fi.values()], textposition='outside',
        textfont=dict(size=10, color=TEXT_SEC)))
    fig.update_layout(**CHART_LAYOUT, title=title, height=380, xaxis_title='Importance')
    return fig

def chart_predictions_scatter(pred_df, x_col, label_col, title, threshold=0.5):
    color_map = {'High': RED, 'Medium': ORANGE, 'Low': GREEN}
    fig = go.Figure()
    for risk in ['High', 'Medium', 'Low']:
        sub = pred_df[pred_df['Risk Level'].astype(str) == risk]
        if sub.empty: continue
        fig.add_trace(go.Scatter(x=sub[x_col], y=sub['Probability'], mode='markers+text',
            text=sub[label_col], textposition='top center', textfont=dict(size=8, color=TEXT_SEC),
            marker=dict(color=color_map[risk], size=10, opacity=0.85, line=dict(color=BG_CARD, width=1)),
            name=f"{risk} Risk"))
    fig.add_hline(y=threshold, line_dash='dash', line_color=TEXT_SEC, line_width=1,
                  annotation_text=f'Threshold ({threshold})', annotation_font_color=TEXT_SEC)
    fig.update_layout(**CHART_LAYOUT, title=title, height=420, xaxis_title=x_col, yaxis_title='Probability')
    fig.update_yaxes(range=[0, 1.1])
    return fig

def chart_market_risk_treemap(pred_df, df):
    mrkt = pred_df.groupby('Market').agg(avg_prob=('Probability', 'mean'),
        flagged=('Prediction', 'sum'), total=('Prediction', 'count')).reset_index()
    mrkt['pct_flagged'] = (mrkt['flagged'] / mrkt['total'] * 100).round(1)
    fig = go.Figure(go.Treemap(labels=mrkt['Market'], parents=[''] * len(mrkt), values=mrkt['total'],
        customdata=np.stack([mrkt['avg_prob'], mrkt['pct_flagged'], mrkt['flagged']], axis=-1),
        hovertemplate='<b>%{label}</b><br>Avg Prob: %{customdata[0]:.2f}<br>Flagged: %{customdata[2]} (%{customdata[1]}%)<extra></extra>',
        marker=dict(colors=mrkt['avg_prob'], colorscale=[[0, GREEN],[0.5, ORANGE],[1, RED]],
                    showscale=True, colorbar=dict(title='Risk', tickfont=dict(color=TEXT_SEC))),
        textfont=dict(color=TEXT_PRI, size=12)))
    fig.update_layout(**CHART_LAYOUT, title='Market Risk Distribution', height=400)
    return fig

def chart_quarterly_trend(df):
    q_data = df.groupby('FQ').agg(obs=('Account', 'count'), accounts=('Account', 'nunique'),
        markets=('Market', 'nunique')).reindex(VALID_QS, fill_value=0).reset_index()
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=q_data['FQ'], y=q_data['obs'], name='Observations', marker_color=ACCENT, opacity=0.7))
    fig.add_trace(go.Scatter(x=q_data['FQ'], y=q_data['accounts'], mode='lines+markers',
        name='Accounts', line=dict(color=GREEN, width=2), marker=dict(size=6)), secondary_y=True)
    fig.add_trace(go.Scatter(x=q_data['FQ'], y=q_data['markets'], mode='lines+markers',
        name='Markets', line=dict(color=ORANGE, width=2), marker=dict(size=6)), secondary_y=True)
    fig.update_layout(**CHART_LAYOUT, title='Quarterly Trend', height=320, barmode='overlay')
    return fig

def chart_root_cause_market_heatmap(df):
    rc_mkts = df.pivot_table(index='Root_Cause_Predicted', columns='Market',
                             values='Account', aggfunc='count', fill_value=0)
    fig = go.Figure(go.Heatmap(z=rc_mkts.values, x=rc_mkts.columns.tolist(), y=rc_mkts.index.tolist(),
        colorscale=[[0, BG_MAIN],[0.5, PURPLE],[1, RED]],
        hovertemplate='%{y}<br>%{x}<br>Count: %{z}<extra></extra>', showscale=True))
    fig.update_layout(**CHART_LAYOUT, title='Root Cause \u00d7 Market Heatmap', height=420)
    fig.update_xaxes(tickangle=-45)
    return fig

def to_excel_bytes(dfs):
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine='openpyxl') as writer:
        for sheet_name, df in dfs.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    return buf.getvalue()

# ═══════════════════════════════════════════════════════════════
#  SIDEBAR
# ═══════════════════════════════════════════════════════════════
with st.sidebar:
    st.markdown(f"""
    <div style="padding:20px 0 12px 0;">
        <div style="font-size:22px; font-weight:700; color:{ACCENT};">\U0001f6e1\ufe0f Compliance</div>
        <div style="font-size:12px; color:{TEXT_SEC}; letter-spacing:1px; margin-top:2px;">Intelligence Platform</div>
        <hr style="border-color:{BORDER}; margin:16px 0;"/>
    </div>
    """, unsafe_allow_html=True)

    st.markdown(f"<div style='font-size:11px; color:{TEXT_SEC}; font-weight:600; margin-bottom:8px;'>\U0001f4c2 DATA UPLOAD</div>", unsafe_allow_html=True)
    uploaded = st.file_uploader("Upload Observation List (.xlsx)", type=['xlsx'], label_visibility='collapsed')

    st.markdown(f"<hr style='border-color:{BORDER};'/>", unsafe_allow_html=True)
    st.markdown(f"<div style='font-size:11px; color:{TEXT_SEC}; font-weight:600; margin-bottom:8px;'>\u2699\ufe0f CONFIGURATION</div>", unsafe_allow_html=True)

    scenario = st.selectbox("Active Scenario",
        ["Scenario 1 \u2014 Recurring Non-Compliance",
         "Scenario 2 \u2014 Systemic Root Causes",
         "Scenario 3 \u2014 Risk Escalation",
         "Scenario 4 \u2014 Root Cause Prediction"])

    selected_models = st.multiselect("Models to Train", list(MODELS_DEF.keys()), default=list(MODELS_DEF.keys()))
    if not selected_models:
        selected_models = list(MODELS_DEF.keys())

    if "Scenario 1" in scenario:
        threshold_s2 = 3
    else:
        threshold_s2 = st.slider("Systemic Threshold (new markets)", 2, 5, 3)

    st.markdown(f"<hr style='border-color:{BORDER};'/>", unsafe_allow_html=True)
    st.markdown(f"<div style='font-size:11px; color:{TEXT_SEC}; font-weight:600; margin-bottom:8px;'>\U0001f3af FILTERS</div>", unsafe_allow_html=True)

    risk_filter = st.multiselect("Show Risk Levels", ['High', 'Medium', 'Low'], default=['High', 'Medium', 'Low'])
    prob_threshold = st.slider("Probability Threshold", 0.1, 0.9, 0.5, 0.05)
    show_raw = st.checkbox("Show Raw Data", value=False)
    show_feat_details = st.checkbox("Show Feature Definitions", value=False)

    st.markdown(f"<hr style='border-color:{BORDER};'/>", unsafe_allow_html=True)
    st.markdown(f"""
    <div style="font-size:11px; color:{TEXT_SEC}; line-height:1.8;">
    \U0001f4c5 Target: <strong style="color:{ACCENT};">2026 Q2</strong><br>
    \U0001f4ca Data: 2024 Q1 \u2192 2026 Q1<br>
    \U0001f916 Models: 5 classifiers<br>
    \U0001f4cf S1: Stratified 5-Fold CV<br>
    \U0001f4cf S2: Leave-One-Out CV
    </div>
    """, unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════════
#  MAIN CONTENT
# ═══════════════════════════════════════════════════════════════
st.markdown(f"""
<div style="background:{BG_CARD}; border:1px solid {BORDER}; border-radius:16px;
            padding:24px 32px; margin-bottom:24px; box-shadow: 0 2px 8px rgba(0,0,0,0.04);">
    <div style="display:flex; align-items:center; gap:16px;">
        <div style="font-size:36px;">\U0001f6e1\ufe0f</div>
        <div>
            <div style="font-size:24px; font-weight:700; color:{TEXT_PRI};">Compliance Intelligence Platform</div>
            <div style="font-size:13px; color:{TEXT_SEC}; margin-top:4px;">Enterprise Risk Prediction \u2014 4 Scenarios \u2014 2026 Q2 Outlook</div>
        </div>
    </div>
</div>
""", unsafe_allow_html=True)

if uploaded is None:
    st.markdown(f"""
    <div style="background:{BG_CARD}; border:2px dashed {BORDER}; border-radius:16px;
                padding:60px; text-align:center; margin:40px 0;">
        <div style="font-size:48px; margin-bottom:16px;">\U0001f4c2</div>
        <div style="font-size:18px; color:{TEXT_PRI}; font-weight:600; margin-bottom:8px;">Upload Your Observation List</div>
        <div style="font-size:13px; color:{TEXT_SEC}; max-width:500px; margin:0 auto; line-height:1.7;">
            Upload <code style="background:{BG_MAIN}; padding:2px 8px; border-radius:6px;
            color:{ACCENT};">Observation List New.xlsx</code> via the sidebar.<br><br>
            Required: <code style="background:{BG_MAIN}; padding:2px 6px; border-radius:4px;
            color:{GREEN}; font-size:11px;">Group \u00b7 Market \u00b7 Country \u00b7 FQ \u00b7 Account \u00b7 Platform \u00b7 Process \u00b7 Domain Area \u00b7 Obs Status \u00b7 Observation \u00b7 Category</code>
        </div>
    </div>
    """, unsafe_allow_html=True)
    st.stop()

with st.spinner("Loading data..."):
    df = load_and_clean(uploaded.getvalue())

m1, m2, m3, m4, m5, m6 = st.columns(6)
m1.metric("Records", f"{len(df):,}")
m2.metric("Accounts", f"{df['Account'].nunique():,}")
m3.metric("Markets", f"{df['Market'].nunique()}")
m4.metric("Quarters", f"{df['FQ'].nunique()}")
m5.metric("Root Causes", f"{df['Root_Cause_Predicted'].nunique()}")
m6.metric("Processes", f"{df['Process'].nunique() if 'Process' in df.columns else 'N/A'}")

st.markdown("<div style='margin:16px 0;'></div>", unsafe_allow_html=True)

tab_overview, tab_eda, tab_features, tab_models, tab_eval, tab_predict, tab_validate, tab_export = st.tabs([
    "\U0001f4ca Overview", "\U0001f50d EDA", "\u2699\ufe0f Features", "\U0001f916 Training",
    "\U0001f4c8 Evaluation", "\U0001f3af Predictions", "\u2705 Validation", "\U0001f4e5 Export"])

# ── TAB 1: OVERVIEW ──
with tab_overview:
    st.caption("Pipeline overview, data quality, business objectives")
    col_a, col_b = st.columns(2)
    with col_a:
        st.markdown(f"""
        <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-left:4px solid {ACCENT};
                    border-radius:12px; padding:20px;">
            <div style="color:{ACCENT}; font-size:13px; font-weight:600; margin-bottom:12px;">SCENARIO 1 \u2014 RECURRING NON-COMPLIANCE</div>
            <table style="width:100%; font-size:13px; border-collapse:collapse;">
                <tr><td style="color:{TEXT_SEC}; padding:6px 8px 6px 0; width:40%;">Target</td><td style="color:{TEXT_PRI};">recurring = 1</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">Approach</td><td>Rolling Window Classification</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">CV Strategy</td><td>Stratified 5-Fold</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">Success</td><td style="color:{GREEN}; font-weight:600;">F1 \u2265 0.50 \u00b7 AUC \u2265 0.80</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">Granularity</td><td>Account \u2192 Market</td></tr>
            </table>
        </div>
        """, unsafe_allow_html=True)
    with col_b:
        st.markdown(f"""
        <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-left:4px solid {GREEN};
                    border-radius:12px; padding:20px;">
            <div style="color:{GREEN}; font-size:13px; font-weight:600; margin-bottom:12px;">SCENARIO 2 \u2014 SYSTEMIC ROOT CAUSES</div>
            <table style="width:100%; font-size:13px; border-collapse:collapse;">
                <tr><td style="color:{TEXT_SEC}; padding:6px 8px 6px 0; width:40%;">Target</td><td style="color:{TEXT_PRI};">systemic = 1 (\u22653 new markets)</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">Root Cause</td><td>Rule-based + KMeans (12 labels)</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">CV Strategy</td><td>Leave-One-Out</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">Success</td><td style="color:{GREEN}; font-weight:600;">Recall \u2265 0.60 \u00b7 AUC \u2265 0.70</td></tr>
                <tr><td style="color:{TEXT_SEC}; padding:6px 0;">Granularity</td><td>Root Cause \u2192 Market Spread</td></tr>
            </table>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("<div style='margin:20px 0;'></div>", unsafe_allow_html=True)

    pipeline_steps = ["\U0001f4c2 Data", "\U0001f50d Quality", "\U0001f4ca EDA", "\u2699\ufe0f Preprocess", "\U0001f916 Train", "\U0001f4c8 Evaluate", "\U0001f680 Deploy"]
    cols = st.columns(len(pipeline_steps))
    for i, (col, step) in enumerate(zip(cols, pipeline_steps)):
        color = GREEN if i == len(pipeline_steps)-1 else ACCENT
        col.markdown(f"""<div style="background:{BG_CARD}; border:1px solid {color}; border-radius:10px;
            padding:12px 8px; text-align:center; font-size:12px; font-weight:500; color:{color};">{step}</div>""", unsafe_allow_html=True)

    st.markdown("<div style='margin:20px 0;'></div>", unsafe_allow_html=True)
    col1, col2 = st.columns(2)
    with col1:
        st.caption("DATA QUALITY")
        missing = df.isnull().sum()
        miss_df = pd.DataFrame({'Column': missing.index, 'Missing': missing.values, 'Pct': (missing/len(df)*100).round(2).values})
        miss_df = miss_df[miss_df['Missing'] > 0]
        if miss_df.empty:
            st.success("\u2705 No missing values")
        else:
            st.dataframe(miss_df, use_container_width=True, hide_index=True)
    with col2:
        st.caption("QUARTER COVERAGE")
        q_cov = df['FQ'].value_counts().reindex(VALID_QS, fill_value=0).reset_index()
        q_cov.columns = ['Quarter', 'Observations']
        fig_qcov = px.bar(q_cov, x='Quarter', y='Observations', color='Observations',
                           color_continuous_scale=[[0, ACCENT],[1, GREEN]])
        fig_qcov.update_layout(**CHART_LAYOUT, height=260, showlegend=False, coloraxis_showscale=False)
        st.plotly_chart(fig_qcov, use_container_width=True)

    if show_raw:
        st.caption("RAW DATA")
        st.dataframe(df.head(200), use_container_width=True)

# ── TAB 2: EDA ──
with tab_eda:
    st.caption("Distributions, patterns, market intelligence")
    st.plotly_chart(chart_quarterly_trend(df), use_container_width=True)

    col1, col2 = st.columns(2)
    with col1:
        mkt_obs = df['Market'].value_counts().reset_index()
        mkt_obs.columns = ['Market', 'Count']
        fig_mkt = px.bar(mkt_obs, x='Count', y='Market', orientation='h', color='Count',
                          color_continuous_scale=[[0, ACCENT],[1, PURPLE]])
        fig_mkt.update_layout(**CHART_LAYOUT, title='Observations per Market', height=380, coloraxis_showscale=False)
        st.plotly_chart(fig_mkt, use_container_width=True)
    with col2:
        rc_dist = df['Root_Cause_Predicted'].value_counts().reset_index()
        rc_dist.columns = ['Root Cause', 'Count']
        fig_rc = px.pie(rc_dist, names='Root Cause', values='Count', color_discrete_sequence=PALETTE, hole=0.45)
        fig_rc.update_layout(**CHART_LAYOUT, title='Root Cause Distribution', height=380)
        st.plotly_chart(fig_rc, use_container_width=True)

    mq_heat = df.groupby(['Market', 'FQ'])['Account'].count().unstack(fill_value=0)
    fig_heat = go.Figure(go.Heatmap(z=mq_heat.values, x=mq_heat.columns.tolist(), y=mq_heat.index.tolist(),
        colorscale=[[0, BG_MAIN],[0.5, ACCENT],[1, GREEN]],
        hovertemplate='%{y}<br>%{x}<br>Obs: %{z}<extra></extra>'))
    fig_heat.update_layout(**CHART_LAYOUT, title='Market \u00d7 Quarter Heatmap', height=420)
    fig_heat.update_xaxes(tickangle=-45)
    st.plotly_chart(fig_heat, use_container_width=True)
    st.plotly_chart(chart_root_cause_market_heatmap(df), use_container_width=True)

    col3, col4 = st.columns(2)
    with col3:
        acc_q = df.groupby('Account')['FQ'].nunique().reset_index()
        acc_q.columns = ['Account', 'Quarters Active']
        fig_aq = px.histogram(acc_q, x='Quarters Active', nbins=9, color_discrete_sequence=[ACCENT])
        fig_aq.update_layout(**CHART_LAYOUT, title='Accounts by Quarters Active', height=300)
        st.plotly_chart(fig_aq, use_container_width=True)
    with col4:
        if 'Process' in df.columns:
            proc = df['Process'].value_counts().head(10).reset_index()
            proc.columns = ['Process', 'Count']
            fig_proc = px.bar(proc, x='Count', y='Process', orientation='h', color='Count',
                               color_continuous_scale=[[0, TEAL],[1, ACCENT]])
            fig_proc.update_layout(**CHART_LAYOUT, title='Top Processes', height=300, coloraxis_showscale=False)
            st.plotly_chart(fig_proc, use_container_width=True)

# ── TAB 3: FEATURES ──
with tab_features:
    st.caption("Rolling-window feature construction \u2014 zero data-leakage")
    feat_tab_s1, feat_tab_s2 = st.tabs(["Scenario 1", "Scenario 2"])

    with feat_tab_s1:
        s1_feat_defs = {
            'streak \u2605': ('0.22', 'Consecutive quarters active before T', 'Momentum'),
            'total_obs': ('0.18', 'Total observations in history', 'Volume'),
            'recent_obs': ('0.15', 'Observations in T-1', 'Recency'),
            'quarters_active': ('0.13', 'Distinct quarters appeared', 'Persistence'),
            'unique_domains': ('0.11', 'Distinct Domain Areas', 'Breadth'),
            'unique_processes': ('0.10', 'Distinct Processes', 'Complexity'),
            'market_enc': ('0.07', 'Encoded Market ID', 'Context'),
            'category_enc': ('0.04', 'Encoded Category ID', 'Profile'),
        }
        for feat, (imp, defn, rationale) in s1_feat_defs.items():
            bar_w = float(imp) * 400
            st.markdown(f"""
            <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-radius:10px; padding:14px 20px; margin-bottom:8px;">
                <div style="display:flex; justify-content:space-between;">
                    <div style="font-size:14px; color:{TEXT_PRI}; font-weight:600;">{feat}</div>
                    <div style="font-size:13px; color:{GREEN}; font-weight:600;">{imp}</div>
                </div>
                <div style="background:{BG_MAIN}; border-radius:4px; height:4px; margin:10px 0;">
                    <div style="background:linear-gradient(90deg, {ACCENT}, {GREEN}); height:4px; border-radius:4px; width:{bar_w}px; max-width:100%;"></div>
                </div>
                <div style="font-size:12px; color:{TEXT_SEC};">{defn} \u2014 <em>{rationale}</em></div>
            </div>
            """, unsafe_allow_html=True)

        if show_feat_details:
            with st.spinner("Building features..."):
                s1_df = build_s1_samples(df)
            st.dataframe(s1_df.head(100), use_container_width=True)
            st.caption(f"Samples: {len(s1_df):,} | Recurring: {s1_df['recurring'].mean()*100:.1f}%")

    with feat_tab_s2:
        s2_feat_defs = {
            'streak': ('Consecutive quarters active', 'Persistence'),
            'lag1_mkts': ('Markets in T-1', 'Immediate spread'),
            'lag2_mkts': ('Markets in T-2', 'Trend context'),
            'trend_mkts': ('lag1 \u2212 lag2', 'Direction'),
            'rolling_mean_mkts': ('Avg markets/quarter', 'Baseline'),
            'rolling_std_mkts': ('Std dev spread', 'Volatility'),
            'curr_markets': ('Current quarter markets', 'Footprint'),
            'total_markets_hist': ('Unique markets ever', 'Reach'),
            'quarters_active': ('Quarters observed', 'Longevity'),
            'unique_processes': ('Distinct processes', 'Complexity'),
            'obs_curr_q': ('Current observations', 'Activity'),
            'total_obs_hist': ('Total observations', 'Volume'),
        }
        for feat, (defn, rationale) in s2_feat_defs.items():
            st.markdown(f"""
            <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-radius:10px; padding:12px 18px; margin-bottom:6px; display:flex; gap:16px; align-items:center;">
                <div style="min-width:200px; font-size:13px; color:{ACCENT}; font-weight:600;">{feat}</div>
                <div><span style="font-size:13px; color:{TEXT_PRI};">{defn}</span> <span style="font-size:12px; color:{TEXT_SEC};">\u2014 {rationale}</span></div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown(f"""
        <div style="background:{BG_CARD}; border:1px solid {ORANGE}; border-radius:10px; padding:16px 20px; margin-top:12px; font-size:13px; color:{TEXT_SEC}; line-height:1.7;">
            <div style="color:{ORANGE}; font-weight:600; margin-bottom:6px;">\u26a0\ufe0f Lag Sparsity Note</div>
            Lag features sparse with 1-2 quarters. Powerful with 3+ quarters (expected end 2027).
        </div>
        """, unsafe_allow_html=True)

# ── TAB 4: TRAINING ──
with tab_models:
    st.caption("Train classifiers with cross-validation")
    run_col, _ = st.columns([1, 3])
    with run_col:
        train_btn = st.button("\U0001f680 Train All Models", type='primary', use_container_width=True)

    if train_btn or 'models_trained' in st.session_state:
        with st.spinner("Training Scenario 1 models..."):
            s1_samples = build_s1_samples(df)
            s1_results, s1_fi, s1_df, s1_sc, s1_best, X_tr, X_tr_sc, X_te, X_te_sc, y_tr, y_te = train_s1_models(s1_samples, selected_models)
            st.session_state.update({'s1_results': s1_results, 's1_fi': s1_fi, 's1_df': s1_df,
                's1_sc': s1_sc, 's1_best': s1_best, 's1_samples': s1_samples})

        with st.spinner("Training Scenario 2 models..."):
            s2_samples = build_s2_samples(df, threshold_s2)
            s2_results, s2_fi, s2_df, s2_sc, s2_best = train_s2_models(s2_samples, selected_models)
            if s2_results:
                st.session_state.update({'s2_results': s2_results, 's2_fi': s2_fi, 's2_df': s2_df,
                    's2_sc': s2_sc, 's2_best': s2_best, 's2_samples': s2_samples})
            else:
                st.warning("Scenario 2: not enough samples.")

        with st.spinner("Training Scenario 3 models..."):
            s3_mq, s3_minmax, s3_low_t, s3_high_t = build_s3_market_q(df)
            s3_results, s3_fi, s3_df, s3_sc, s3_best = train_s3_models(s3_mq)
            if s3_results:
                st.session_state.update({'s3_results': s3_results, 's3_fi': s3_fi, 's3_df': s3_df,
                    's3_sc': s3_sc, 's3_best': s3_best, 's3_mq': s3_mq})
            else:
                st.warning("Scenario 3: not enough escalation events.")

        with st.spinner("Training Scenario 4 models..."):
            s4_results, s4_model, s4_tfidf, s4_le, s4_best = train_s4_models(df)
            if s4_results:
                st.session_state.update({'s4_results': s4_results, 's4_model': s4_model,
                    's4_tfidf': s4_tfidf, 's4_le': s4_le, 's4_best': s4_best})
            else:
                st.warning("Scenario 4: not enough labelled data.")

        st.session_state['models_trained'] = True

        # Display results for the currently selected scenario
        if "Scenario 1" in scenario:
            results, best_name, cv_key = s1_results, s1_best, 'cv_f1'
        elif "Scenario 2" in scenario:
            results = st.session_state.get('s2_results', {})
            best_name = st.session_state.get('s2_best', '')
            cv_key = 'loo_f1'
        elif "Scenario 3" in scenario:
            results = st.session_state.get('s3_results', {})
            best_name = st.session_state.get('s3_best', '')
            cv_key = 'loo_f1'
        elif "Scenario 4" in scenario:
            results = st.session_state.get('s4_results', {})
            best_name = st.session_state.get('s4_best', '')
            cv_key = 'macro_f1'
        else:
            results, best_name, cv_key = s1_results, s1_best, 'cv_f1'

        if not results:
            st.error("Training failed for selected scenario.")
            st.stop()

        st.success(f"\u2705 Best model: **{best_name}** (Composite: {results[best_name]['composite']:.4f})")

        st.caption("MODEL SUMMARY")
        model_cols = st.columns(len(results))
        for col, (name, r) in zip(model_cols, results.items()):
            is_best = (name == best_name)
            border_c = GREEN if is_best else BORDER
            badge = f'<div style="font-size:10px; color:{GREEN}; font-weight:700; margin-bottom:4px;">\u2b50 BEST</div>' if is_best else '<div style="margin-bottom:14px;"></div>'
            col.markdown(f"""
            <div style="background:{BG_CARD}; border:1px solid {border_c}; border-radius:12px; padding:16px; text-align:center;">
                {badge}
                <div style="font-size:12px; color:{TEXT_PRI}; font-weight:600; margin-bottom:10px;">{name}</div>
                <div style="font-size:20px; color:{ACCENT}; font-weight:700;">{r['composite']:.4f}</div>
                <div style="font-size:10px; color:{TEXT_SEC};">Composite</div>
                <hr style="border-color:{BORDER}; margin:10px 0;"/>
                <div style="font-size:11px; color:{TEXT_SEC};">F1: <strong>{r.get(cv_key,0):.3f}</strong></div>
                <div style="font-size:11px; color:{TEXT_SEC};">AUC: <strong>{r.get('roc_auc', r.get('loo_auc',0)):.3f}</strong></div>
                <div style="font-size:11px; color:{TEXT_SEC};">Recall: <strong>{r.get('recall', r.get('loo_recall',0)):.3f}</strong></div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown(f"""
        <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-radius:10px; padding:14px 20px; margin-top:16px; font-size:13px; color:{TEXT_SEC};">
            <strong style="color:{TEXT_PRI};">Composite</strong> = <span style="color:{ACCENT};">AUC\u00d70.5</span> + <span style="color:{GREEN};">Recall\u00d70.3</span> + <span style="color:{ORANGE};">F1\u00d70.2</span>
        </div>
        """, unsafe_allow_html=True)
    else:
        st.info("\U0001f446 Click **Train All Models** to begin")

# ── TAB 5: EVALUATION ──
with tab_eval:
    st.caption("ROC curves, confusion matrices, feature importance")
    if "Scenario 1" in scenario:
        res_key, fi_key, best_key = 's1_results', 's1_fi', 's1_best'
        metric_cols = ['cv_f1', 'roc_auc', 'precision', 'recall', 'f1']
        scen_label = "Scenario 1"
    else:
        res_key, fi_key, best_key = 's2_results', 's2_fi', 's2_best'
        metric_cols = ['loo_f1', 'loo_recall', 'loo_auc', 'composite']
        scen_label = "Scenario 2"

    if res_key not in st.session_state:
        st.info("\U0001f446 Train models first")
    else:
        results = st.session_state[res_key]
        feat_imp = st.session_state[fi_key]
        best_name = st.session_state[best_key]
        st.plotly_chart(chart_model_comparison(results, metric_cols, scen_label), use_container_width=True)
        col1, col2 = st.columns(2)
        with col1:
            st.plotly_chart(chart_roc_curves(results, f"ROC \u2014 {scen_label}"), use_container_width=True)
        with col2:
            st.plotly_chart(chart_feature_importance(feat_imp, f"Features \u2014 {best_name}"), use_container_width=True)
        st.caption("CONFUSION MATRICES")
        cm_cols = st.columns(min(len(results), 5))
        for col, (name, r) in zip(cm_cols, results.items()):
            col.plotly_chart(chart_confusion_matrix(r['cm'], name), use_container_width=True)

        rows = []
        for name, r in results.items():
            row = {'Model': name, 'Composite': f"{r['composite']:.4f}", 'Best': '\u2b50' if name == best_name else ''}
            for k in metric_cols:
                if k in r: row[k] = f"{r[k]:.4f}"
            rows.append(row)
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)

# ── TAB 6: PREDICTIONS ──
with tab_predict:
    st.caption("2026 Q2 predictions")
    if 'models_trained' not in st.session_state:
        st.info("\U0001f446 Train models first")
    else:
        pred_s1_tab, pred_s2_tab, pred_s3_tab, pred_s4_tab = st.tabs(["S1: Recurring", "S2: Systemic", "S3: Escalation", "S4: Root Cause"])
        with pred_s1_tab:
            if 's1_results' not in st.session_state:
                st.info("Train Scenario 1 first")
            else:
                s1_model_sel = st.selectbox("Model", list(st.session_state['s1_results'].keys()),
                    index=list(st.session_state['s1_results'].keys()).index(st.session_state['s1_best']), key='s1_pred_model')
                with st.spinner("Predicting..."):
                    s1_pred = predict_s1(df, s1_model_sel, st.session_state['s1_results'], st.session_state['s1_sc'], st.session_state['s1_samples'])

                s1_pred['Prediction'] = (s1_pred['Probability'] >= prob_threshold).astype(int)
                s1_pred['Risk Level'] = pd.cut(s1_pred['Probability'],
                    bins=[0, prob_threshold * 0.6, prob_threshold, 1.0],
                    labels=['Low', 'Medium', 'High'])

                mc1, mc2, mc3, mc4 = st.columns(4)
                mc1.metric("Accounts", len(s1_pred))
                mc2.metric(f"Flagged (\u2265{prob_threshold})", int(s1_pred['Prediction'].sum()))
                mc3.metric("High Risk", int((s1_pred['Risk Level'] == 'High').sum()))
                mc4.metric("Avg Prob", f"{s1_pred['Probability'].mean():.3f}")
                st.plotly_chart(chart_predictions_scatter(s1_pred, 'streak', 'Account', 'Account Recurrence Risk \u2014 2026 Q2', prob_threshold), use_container_width=True)
                st.plotly_chart(chart_market_risk_treemap(s1_pred, df), use_container_width=True)
                col_a, col_b = st.columns(2)
                with col_a:
                    st.caption("MARKET ROLLUP")
                    mkt_roll = s1_pred.groupby('Market').agg(Accounts=('Account','count'), Flagged=('Prediction','sum'), Avg_Prob=('Probability','mean')).reset_index().sort_values('Avg_Prob', ascending=False)
                    mkt_roll['Avg_Prob'] = mkt_roll['Avg_Prob'].round(3)
                    st.dataframe(mkt_roll, use_container_width=True, hide_index=True)
                with col_b:
                    st.caption("TOP 20 HIGH-RISK")
                    st.dataframe(s1_pred.head(20)[['Account','Market','Probability','streak','total_obs','Risk Level']], use_container_width=True, hide_index=True)

        with pred_s2_tab:
            if 's2_results' not in st.session_state:
                st.info("Train Scenario 2 first")
            else:
                s2_model_sel = st.selectbox("Model", list(st.session_state['s2_results'].keys()),
                    index=list(st.session_state['s2_results'].keys()).index(st.session_state['s2_best']), key='s2_pred_model')
                with st.spinner("Predicting..."):
                    s2_pred = predict_s2(df, s2_model_sel, st.session_state['s2_results'], st.session_state['s2_sc'])
                if s2_pred.empty:
                    st.warning("No predictions")
                else:
                    s2_pred['Systemic'] = (s2_pred['Probability'] >= prob_threshold).astype(int)
                    s2_pred['Risk Level'] = pd.cut(s2_pred['Probability'],
                        bins=[0, prob_threshold * 0.6, prob_threshold, 1.0],
                        labels=['Low', 'Medium', 'High'])

                    mc1, mc2, mc3, mc4 = st.columns(4)
                    mc1.metric("Root Causes", len(s2_pred))
                    mc2.metric(f"Systemic (\u2265{prob_threshold})", int(s2_pred['Systemic'].sum()))
                    mc3.metric("High Risk", int((s2_pred['Risk Level'] == 'High').sum()))
                    mc4.metric("Max Prob", f"{s2_pred['Probability'].max():.3f}")
                    st.plotly_chart(chart_predictions_scatter(s2_pred, 'streak', 'Root_Cause', 'Systemic Root Cause Risk \u2014 2026 Q2', prob_threshold), use_container_width=True)
                    col_a, col_b = st.columns(2)
                    with col_a:
                        st.caption("SYSTEMIC RISK RANKING")
                        st.dataframe(s2_pred[['Root_Cause','Probability','curr_markets','streak','Risk Level','Systemic']], use_container_width=True, hide_index=True)
                    with col_b:
                        fig_s2b = px.bar(s2_pred.head(12), x='Probability', y='Root_Cause', orientation='h', color='Probability',
                                          color_continuous_scale=[[0, GREEN],[0.5, ORANGE],[1, RED]])
                        fig_s2b.add_vline(x=prob_threshold, line_dash='dash', line_color=TEXT_SEC,
                            annotation_text=f'Threshold: {prob_threshold}', annotation_font_color=TEXT_SEC)
                        fig_s2b.update_layout(**CHART_LAYOUT, title='Root Cause Risk Probabilities', height=380, coloraxis_showscale=False)
                        st.plotly_chart(fig_s2b, use_container_width=True)

        # ── S3: Risk Escalation Predictions ──
        with pred_s3_tab:
            if 's3_results' not in st.session_state:
                st.info("Train Scenario 3 first")
            else:
                s3_model_sel = st.selectbox("Model", list(st.session_state['s3_results'].keys()),
                    index=list(st.session_state['s3_results'].keys()).index(st.session_state['s3_best']), key='s3_pred_model')
                s3_pred = predict_s3(st.session_state['s3_mq'], s3_model_sel,
                                      st.session_state['s3_results'], st.session_state['s3_sc'], VALID_QS)
                if s3_pred.empty:
                    st.warning("No predictions for current quarter")
                else:
                    s3_pred['Predicted_Escalation'] = (s3_pred['Probability'] >= prob_threshold).astype(int)

                    mc1, mc2, mc3, mc4 = st.columns(4)
                    mc1.metric("Markets Scored", len(s3_pred))
                    mc2.metric(f"At Risk (\u2265{prob_threshold})", int(s3_pred['Predicted_Escalation'].sum()))
                    mc3.metric("Current High", int((s3_pred['Risk_Level'] == 'High').sum()))
                    mc4.metric("Max Prob", f"{s3_pred['Probability'].max():.3f}")

                    # Bar chart: escalation probability by market
                    s3_sorted = s3_pred.sort_values('Probability', ascending=True)
                    bar_colors = [RED if p >= 0.7 else ORANGE if p >= prob_threshold else GREEN
                                  for p in s3_sorted['Probability']]
                    fig_s3 = go.Figure(go.Bar(
                        x=s3_sorted['Probability'], y=s3_sorted['Market'],
                        orientation='h', marker_color=bar_colors))
                    fig_s3.add_vline(x=prob_threshold, line_dash='dash', line_color=TEXT_SEC,
                        annotation_text=f'Threshold ({prob_threshold})', annotation_font_color=TEXT_SEC)
                    fig_s3.update_layout(**CHART_LAYOUT, title='Escalation Probability by Market \u2014 2026 Q2',
                                          height=max(300, len(s3_pred) * 28))
                    st.plotly_chart(fig_s3, use_container_width=True)

                    col_a, col_b = st.columns(2)
                    with col_a:
                        st.caption("MARKETS AT RISK")
                        at_risk = s3_pred[s3_pred['Predicted_Escalation'] == 1]
                        if at_risk.empty:
                            st.info("No markets predicted to escalate at this threshold")
                        else:
                            st.dataframe(at_risk[['Market','Risk_Level','Enforcement_Risk_Score','trend_score','Probability']].round(3),
                                         use_container_width=True, hide_index=True)
                    with col_b:
                        st.caption("ALL MARKETS")
                        st.dataframe(s3_pred[['Market','Risk_Level','Enforcement_Risk_Score','Probability','Predicted_Escalation']].round(3),
                                     use_container_width=True, hide_index=True)

        # ── S4: Root Cause Prediction ──
        with pred_s4_tab:
            if 's4_model' not in st.session_state:
                st.info("Train Scenario 4 first")
            else:
                st.markdown(f"""
                <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-left:4px solid {PURPLE};
                            border-radius:12px; padding:20px; margin-bottom:20px;">
                    <div style="color:{PURPLE}; font-size:13px; font-weight:600; margin-bottom:8px;">
                        SCENARIO 4 \u2014 ROOT CAUSE PREDICTION
                    </div>
                    <div style="font-size:13px; color:{TEXT_SEC};">
                        Enter a new audit observation text below. The trained model ({st.session_state['s4_best']})
                        will predict the most likely root cause category.
                    </div>
                </div>
                """, unsafe_allow_html=True)

                obs_text = st.text_area("Enter audit observation text:", height=120,
                    placeholder="e.g., The firewall rules were not reviewed as per the quarterly review schedule...")

                if obs_text.strip():
                    tfidf = st.session_state['s4_tfidf']
                    model = st.session_state['s4_model']
                    le = st.session_state['s4_le']

                    X_input = tfidf.transform([obs_text])
                    pred_idx = model.predict(X_input)[0]
                    pred_label = le.inverse_transform([pred_idx])[0]

                    if hasattr(model, 'predict_proba'):
                        probs = model.predict_proba(X_input)[0]
                        top_indices = probs.argsort()[::-1][:5]
                        top_results = [(le.inverse_transform([i])[0], probs[i]) for i in top_indices]
                    else:
                        top_results = [(pred_label, 1.0)]

                    st.markdown(f"""
                    <div style="background:{BG_CARD}; border:1px solid {GREEN}; border-radius:12px;
                                padding:20px; margin:16px 0;">
                        <div style="font-size:11px; color:{TEXT_SEC}; text-transform:uppercase; margin-bottom:8px;">Predicted Root Cause</div>
                        <div style="font-size:24px; font-weight:700; color:{GREEN};">{pred_label}</div>
                        <div style="font-size:13px; color:{TEXT_SEC}; margin-top:4px;">Confidence: {top_results[0][1]*100:.1f}%</div>
                    </div>
                    """, unsafe_allow_html=True)

                    # Show top 5 probabilities
                    st.caption("TOP 5 PREDICTIONS")
                    for rank, (label, prob) in enumerate(top_results, 1):
                        bar_w = prob * 100
                        color = GREEN if rank == 1 else ACCENT if prob > 0.1 else TEXT_SEC
                        st.markdown(f"""
                        <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-radius:8px;
                                    padding:10px 16px; margin-bottom:6px;">
                            <div style="display:flex; justify-content:space-between;">
                                <span style="font-size:13px; color:{TEXT_PRI}; font-weight:{'600' if rank==1 else '400'};">{rank}. {label}</span>
                                <span style="font-size:13px; color:{color}; font-weight:600;">{prob*100:.1f}%</span>
                            </div>
                            <div style="background:{BG_MAIN}; border-radius:3px; height:4px; margin-top:8px;">
                                <div style="background:{color}; height:4px; border-radius:3px; width:{bar_w}%;"></div>
                            </div>
                        </div>
                        """, unsafe_allow_html=True)

                # Show model performance summary
                if st.session_state.get('s4_results'):
                    with st.expander("\U0001f50d Model Performance"):
                        s4r = st.session_state['s4_results']
                        rows = [{'Model': n, 'Macro F1': f"{r['macro_f1']:.3f}", 'Accuracy': f"{r['accuracy']:.3f}",
                                 'Best': '\u2b50' if n == st.session_state['s4_best'] else ''}
                                for n, r in s4r.items()]
                        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)

# ── TAB 7: VALIDATION ──
with tab_validate:
    st.caption("Threshold tuning and quality review")
    if 's1_results' not in st.session_state and 's2_results' not in st.session_state:
        st.info("\U0001f446 Train models first")
    else:
        val_s1, val_s2 = st.tabs(["Scenario 1", "Scenario 2"])
        with val_s1:
            if 's1_results' not in st.session_state:
                st.info("Train Scenario 1 first")
            else:
                results = st.session_state['s1_results']
                best = st.session_state['s1_best']
                thresholds = np.arange(0.3, 0.8, 0.05)
                s1_samples = st.session_state['s1_samples']
                df_v = s1_samples.dropna(subset=S1_FEATURES + ['recurring'])
                X_v = df_v[S1_FEATURES].values
                y_v = df_v['recurring'].values
                _, X_te_v, _, y_te_v = train_test_split(X_v, y_v, test_size=0.2, random_state=42, stratify=y_v)
                sc_v = StandardScaler().fit(X_v[:int(len(X_v)*0.8)])
                X_te_sc_v = sc_v.transform(X_te_v)
                mdl_v = results[best]['model']
                X_use = X_te_sc_v if best in SCALED_MODELS else X_te_v
                probas_v = mdl_v.predict_proba(X_use)[:, 1]

                thresh_data = []
                for t in thresholds:
                    preds_t = (probas_v >= t).astype(int)
                    thresh_data.append({'Threshold': round(t, 2),
                        'Precision': precision_score(y_te_v, preds_t, zero_division=0),
                        'Recall': recall_score(y_te_v, preds_t, zero_division=0),
                        'F1': f1_score(y_te_v, preds_t, zero_division=0),
                        'Flagged': preds_t.sum()})
                thresh_df = pd.DataFrame(thresh_data)

                fig_thresh = go.Figure()
                for metric, color in [('Precision', ACCENT), ('Recall', GREEN), ('F1', ORANGE)]:
                    fig_thresh.add_trace(go.Scatter(x=thresh_df['Threshold'], y=thresh_df[metric],
                        mode='lines+markers', name=metric, line=dict(color=color, width=2)))
                fig_thresh.add_vline(x=prob_threshold, line_dash='dash', line_color=RED, line_width=2,
                    annotation_text=f'Current: {prob_threshold}', annotation_font_color=RED)
                fig_thresh.update_layout(**CHART_LAYOUT, title='Threshold Sensitivity', height=350,
                    xaxis_title='Threshold', yaxis_title='Score')
                st.plotly_chart(fig_thresh, use_container_width=True)

                col_v1, col_v2 = st.columns(2)
                with col_v1:
                    st.dataframe(thresh_df.style.format({'Precision': '{:.3f}', 'Recall': '{:.3f}', 'F1': '{:.3f}', 'Threshold': '{:.2f}'}),
                                 use_container_width=True, hide_index=True)
                with col_v2:
                    fig_prob = go.Figure()
                    fig_prob.add_trace(go.Histogram(x=probas_v[y_te_v==0], name='Non-Recurring', marker_color=ACCENT, opacity=0.7, nbinsx=20))
                    fig_prob.add_trace(go.Histogram(x=probas_v[y_te_v==1], name='Recurring', marker_color=RED, opacity=0.7, nbinsx=20))
                    fig_prob.add_vline(x=prob_threshold, line_dash='dash', line_color=ORANGE)
                    fig_prob.update_layout(**CHART_LAYOUT, title='Probability Distribution', height=320, barmode='overlay')
                    st.plotly_chart(fig_prob, use_container_width=True)

                criteria = [
                    ('F1 \u2265 0.50', results[best].get('f1', 0) >= 0.50, f"{results[best].get('f1',0):.4f}"),
                    ('AUC \u2265 0.80', results[best].get('roc_auc',0) >= 0.80, f"{results[best].get('roc_auc',0):.4f}"),
                ]
                crit_cols = st.columns(len(criteria))
                for col, (label, passed, val) in zip(crit_cols, criteria):
                    icon = "\u2705" if passed else "\u26a0\ufe0f"
                    color = GREEN if passed else ORANGE
                    col.markdown(f"""
                    <div style="background:{BG_CARD}; border:1px solid {color}; border-radius:10px; padding:16px; text-align:center;">
                        <div style="font-size:24px;">{icon}</div>
                        <div style="font-size:13px; color:{TEXT_PRI}; margin:6px 0;">{label}</div>
                        <div style="font-size:20px; color:{color}; font-weight:700;">{val}</div>
                    </div>
                    """, unsafe_allow_html=True)

        with val_s2:
            if 's2_results' not in st.session_state:
                st.info("Train Scenario 2 first")
            else:
                results2 = st.session_state['s2_results']
                best2 = st.session_state['s2_best']
                r = results2[best2]
                criteria2 = [
                    ('Recall \u2265 0.60', r.get('loo_recall',0) >= 0.60, f"{r.get('loo_recall',0):.4f}"),
                    ('AUC \u2265 0.70', r.get('loo_auc',0) >= 0.70, f"{r.get('loo_auc',0):.4f}"),
                    ('Composite \u2265 0.60', r.get('composite',0) >= 0.60, f"{r.get('composite',0):.4f}"),
                ]
                crit_cols2 = st.columns(len(criteria2))
                for col, (label, passed, val) in zip(crit_cols2, criteria2):
                    icon = "\u2705" if passed else "\u26a0\ufe0f"
                    color = GREEN if passed else ORANGE
                    col.markdown(f"""
                    <div style="background:{BG_CARD}; border:1px solid {color}; border-radius:10px; padding:16px; text-align:center;">
                        <div style="font-size:24px;">{icon}</div>
                        <div style="font-size:13px; color:{TEXT_PRI}; margin:6px 0;">{label}</div>
                        <div style="font-size:20px; color:{color}; font-weight:700;">{val}</div>
                    </div>
                    """, unsafe_allow_html=True)

                y_all = r['y_all']
                y_proba = r['y_proba']
                fig_p2 = go.Figure()
                fig_p2.add_trace(go.Histogram(x=y_proba[y_all==0], name='Non-Systemic', marker_color=ACCENT, opacity=0.7, nbinsx=15))
                fig_p2.add_trace(go.Histogram(x=y_proba[y_all==1], name='Systemic', marker_color=RED, opacity=0.7, nbinsx=15))
                fig_p2.add_vline(x=0.5, line_dash='dash', line_color=ORANGE)
                fig_p2.update_layout(**CHART_LAYOUT, title='LOO Probability Distribution', height=300, barmode='overlay')
                st.plotly_chart(fig_p2, use_container_width=True)

# ── TAB 8: EXPORT ──
with tab_export:
    st.caption("Export predictions and results to Excel")
    export_sheets = {}
    raw_cols = [c for c in ['Market','Account','FQ','Root_Cause_Predicted','Obs Status','Category','Process'] if c in df.columns]
    export_sheets['Raw_Data'] = df[raw_cols].copy()

    if 's1_results' in st.session_state and 's1_samples' in st.session_state:
        export_sheets['S1_Feature_Matrix'] = st.session_state['s1_samples']
        results1 = st.session_state['s1_results']
        best1 = st.session_state['s1_best']
        rows1 = [{'Model': n, 'Is_Best': n == best1, 'Composite': round(r['composite'], 4),
                   'CV_F1': round(r.get('cv_f1',0), 4), 'ROC_AUC': round(r.get('roc_auc',0), 4),
                   'Precision': round(r.get('precision',0), 4), 'Recall': round(r.get('recall',0), 4),
                   'F1': round(r.get('f1',0), 4)} for n, r in results1.items()]
        export_sheets['S1_Model_Results'] = pd.DataFrame(rows1)
        try:
            s1_pred_exp = predict_s1(df, best1, results1, st.session_state['s1_sc'], st.session_state['s1_samples'])
            export_sheets['S1_Predictions'] = s1_pred_exp[['Account','Market','Probability','Prediction','Risk Level','streak','total_obs','quarters_active']]
        except: pass
        export_sheets['S1_Feature_Imp'] = pd.DataFrame({'Feature': list(st.session_state['s1_fi'].keys()),
            'Importance': list(st.session_state['s1_fi'].values())}).sort_values('Importance', ascending=False)

    if 's2_results' in st.session_state and 's2_samples' in st.session_state:
        export_sheets['S2_Feature_Matrix'] = st.session_state['s2_samples']
        results2 = st.session_state['s2_results']
        best2 = st.session_state['s2_best']
        rows2 = [{'Model': n, 'Is_Best': n == best2, 'Composite': round(r['composite'], 4),
                   'LOO_F1': round(r.get('loo_f1',0), 4), 'LOO_Recall': round(r.get('loo_recall',0), 4),
                   'LOO_AUC': round(r.get('loo_auc',0), 4)} for n, r in results2.items()]
        export_sheets['S2_Model_Results'] = pd.DataFrame(rows2)
        try:
            s2_pred_exp = predict_s2(df, best2, results2, st.session_state['s2_sc'])
            if not s2_pred_exp.empty:
                export_sheets['S2_Predictions'] = s2_pred_exp[['Root_Cause','Probability','Systemic','curr_markets','streak','Risk Level','trend_mkts']]
        except: pass
        export_sheets['S2_Feature_Imp'] = pd.DataFrame({'Feature': list(st.session_state['s2_fi'].keys()),
            'Importance': list(st.session_state['s2_fi'].values())}).sort_values('Importance', ascending=False)

    if 's3_results' in st.session_state and 's3_mq' in st.session_state:
        export_sheets['S3_Market_Q'] = st.session_state['s3_mq'][['Market','FQ','Enforcement_Risk_Score','Risk_Level','Escalation_Flag']].copy()
        try:
            s3_pred_exp = predict_s3(st.session_state['s3_mq'], st.session_state['s3_best'],
                                      st.session_state['s3_results'], st.session_state['s3_sc'], VALID_QS)
            if not s3_pred_exp.empty:
                export_sheets['S3_Predictions'] = s3_pred_exp[['Market','Risk_Level','Enforcement_Risk_Score','Probability']].round(4)
        except: pass
        if st.session_state.get('s3_fi'):
            export_sheets['S3_Feature_Imp'] = pd.DataFrame({'Feature': list(st.session_state['s3_fi'].keys()),
                'Importance': list(st.session_state['s3_fi'].values())}).sort_values('Importance', ascending=False)

    if 's4_results' in st.session_state:
        s4r = st.session_state['s4_results']
        rows4 = [{'Model': n, 'Macro_F1': round(r['macro_f1'], 4), 'Accuracy': round(r['accuracy'], 4)}
                 for n, r in s4r.items()]
        export_sheets['S4_Model_Results'] = pd.DataFrame(rows4)

    st.caption("AVAILABLE SHEETS")
    for sheet in export_sheets:
        st.markdown(f"""
        <div style="background:{BG_CARD}; border:1px solid {BORDER}; border-radius:8px; padding:10px 16px; margin-bottom:6px; display:flex; justify-content:space-between;">
            <span style="font-size:13px; color:{TEXT_PRI};">\U0001f4c4 {sheet}</span>
            <span style="font-size:12px; color:{TEXT_SEC};">{len(export_sheets[sheet]):,} rows</span>
        </div>
        """, unsafe_allow_html=True)

    if len(export_sheets) > 1:
        excel_bytes = to_excel_bytes(export_sheets)
        st.download_button(label="\U0001f4e5 Download Report (.xlsx)", data=excel_bytes,
            file_name="Compliance_Report_2026Q2.xlsx",
            mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            type='primary', use_container_width=True)
        st.success(f"\u2705 Ready \u2014 {len(export_sheets)} sheets, {sum(len(v) for v in export_sheets.values()):,} rows")
    else:
        st.info("Train models to generate exports")

    st.markdown(f"""
    <div style="margin-top:40px; padding:16px; border-top:1px solid {BORDER}; text-align:center; font-size:11px; color:{TEXT_SEC};">
        \U0001f6e1\ufe0f Compliance Intelligence Platform \u2014 Enterprise Edition \u2014 2026 Q2
    </div>
    """, unsafe_allow_html=True)



Overwriting /content/compliance_dashboard.py


---
## 🚀 Cell 4 — Launch Dashboard

Paste your **ngrok token** below and run this cell.


In [16]:
# ─── Launch Dashboard ──────────────────────────────────────────
# Get your free token: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = "39iTYzQWDA3dovf0bvGMtP8cAll_86UQuqDVhu8Sbr5qj4pZ8"  # ← PASTE YOUR TOKEN HERE

import subprocess, threading, time, os
from pyngrok import ngrok, conf

if not NGROK_TOKEN.strip():
    print("⚠️  No ngrok token set.")
    print()
    print("  1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("  2. Copy your token and paste it into NGROK_TOKEN above")
    print("  3. Re-run this cell")
else:
    app_path = "/content/compliance_dashboard.py"

    if not os.path.exists(app_path):
        print("⚠️  App file not found. Run Cell 3 first.")
    else:
        os.system("pkill -f streamlit 2>/dev/null; pkill -f ngrok 2>/dev/null")
        time.sleep(1)

        conf.get_default().auth_token = NGROK_TOKEN

        def run_streamlit():
            subprocess.run([
                "streamlit", "run", app_path,
                "--server.port", "8501",
                "--server.headless", "true",
                "--server.enableCORS", "false",
                "--server.enableXsrfProtection", "false",
                "--browser.gatherUsageStats", "false",
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        t = threading.Thread(target=run_streamlit, daemon=True)
        t.start()
        print("🚀 Starting Streamlit...")
        time.sleep(5)

        tunnel = ngrok.connect(8501, bind_tls=True)
        url = tunnel.public_url

        print()
        print("━" * 56)
        print("  🛡️  COMPLIANCE INTELLIGENCE PLATFORM")
        print("━" * 56)
        print(f"  ✅ Dashboard is LIVE")
        print(f"  🔗 Open → {url}")
        print("━" * 56)
        print("  📂 Upload your .xlsx in the sidebar")
        print("  ⏳ Keep this cell running")
        print("━" * 56)


🚀 Starting Streamlit...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🛡️  COMPLIANCE INTELLIGENCE PLATFORM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ Dashboard is LIVE
  🔗 Open → https://gunner-claylike-fleeringly.ngrok-free.dev
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📂 Upload your .xlsx in the sidebar
  ⏳ Keep this cell running
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


---
## 📝 Notes

| Item | Detail |
|------|--------|
| **ngrok token** | Free at [dashboard.ngrok.com](https://dashboard.ngrok.com) |
| **Data file** | Upload `.xlsx` via the dashboard sidebar |
| **Session** | Stays live while Colab is active |
| **Restart** | Re-run Cell 4 only |
| **Timeout** | Free Colab disconnects after ~90 min idle |

### Required Excel Columns

`Group` · `Market` · `Country` · `FQ` · `Account` · `Platform` · `Process` · `Domain Area` · `Obs Status` · `Observation` · `Category`


In [14]:
!pkill streamlit
!pkill ngrok # Added to kill existing ngrok processes
!streamlit run app.py &>/dev/null &